This notebook contains Python code for reproducing the results in our paper on using a large language model to simplify textbook content while preserving meaning and improving readability:

Johnson, B. G., Jerome, B., Dittel, J. S., & Van Campenhout, R. (2025). Improving textbook readability through AI simplification: Readability improvements and meaning preservation. In *Proceedings of the Sixth Workshop on Intelligent Textbooks at the 24th International Conference on Artificial Intelligence in Education*. CEUR Workshop Proceedings. \*\*\*[https://doi.org/PLACEHOLDER\_DOI](https://doi.org/PLACEHOLDER_DOI)

This paper was presented at [AIED 2025](https://aied2025.itd.cnr.it/) as part of the [Sixth Workshop on Intelligent Textbooks (iTextbooks)](https://intextbooks.science.uu.nl/workshop2025/).

Results are presented in the order they occur, organized by the paper's sections. For each result, an excerpt from the paper is given followed by code to compute the result from the data set provided. Example:

>The dataset contains 54,371 events generated by 11,689 students across 2,082 distinct textbooks.

```len( events ), events.student_id.nunique(), events.textbook_id.nunique()```

Please refer to the paper for additional context.

In [1]:
import pandas as pd
import scipy.stats

## Read dataset of simplification events

In [2]:
events = pd.read_parquet( 'events.parquet' )
events.head()

,timestamp,student_id,textbook_id,subject,selected,simplified,rd_sel_fkgl,rd_sim_fkgl,rd_sel_fre,rd_sim_fre,...,sem_sel_words,sem_sim_words,rd_fkgl_difference,rd_fre_difference,lex_content_log_p_difference,lex_chars_per_word_difference,syn_dep_depth_difference,syn_words_per_sentence_difference,sem_cosine_similarity,sem_compression_ratio
0,2024-09-03 18:24:26,ENKNSSSXJP4QNNPE4DBP,9781544385563,Language Arts & Disciplines,0.1 Describe relevant theories of communicatio...,0.1 Describe important communication theories ...,29.197692,13.335556,-21.652692,26.170000,...,52,45,-15.862137,47.822692,0.390713,-0.405556,0.158200,-7.000000,0.963072,0.865385
1,2024-09-03 22:25:52,AZVR2SR3JNT58C43UB45,9781544385563,Language Arts & Disciplines,Theories exist to provide a framework for publ...,Theories help public relations professionals m...,13.262087,10.178807,32.752478,42.390316,...,92,57,-3.083280,9.637838,0.469336,-0.019808,-0.650350,-7.200000,0.977973,0.619565
2,2024-09-06 21:00:52,BH5MJB7SKRE43Z7QCUEA,9781071845226,Social Science,Much of the study of women and crime focuses o...,Many studies about women and crime focus on ho...,14.606315,10.890278,33.311856,45.482500,...,193,144,-3.716037,12.170644,0.531052,-0.071809,-0.489959,-8.138889,0.963286,0.746114
3,2024-09-06 21:07:01,BH5MJB7SKRE43Z7QCUEA,9781071845226,Social Science,To develop an understanding of how often women...,To understand how often women commit crimes or...,17.138599,10.029849,24.602864,54.545625,...,205,141,-7.108750,29.942761,0.292410,-0.569637,-1.385672,-9.428571,0.911901,0.687805
4,2024-09-07 23:29:17,NJ8WDEYTGP8YV7QTFSNM,9781544385563,Language Arts & Disciplines,Considered a monumental study in public relati...,The Excellence Study is a major work in public...,24.933934,13.188723,-11.269536,28.518061,...,244,173,-11.745211,39.787596,1.038542,-0.104339,-1.294835,-25.575758,0.910702,0.709016


## 2. Method

### 2.2. Data Collection and Analysis

>The dataset contains 54,371 events generated by 11,689 students across 2,082 distinct textbooks.

In [3]:
len( events ), events.student_id.nunique(), events.textbook_id.nunique()

(54371, 11689, 2082)

>Using the BISAC major subject heading classification for the textbooks [15], the top subject domains as a percentage of the data were Social Science (29.7%), Political Science (16.1%), and Psychology (13.8%).

In [4]:
events.subject.value_counts( normalize=True ).apply( lambda p: f'{p:.1%}' )

subject
Social Science                       29.7%
Political Science                    16.1%
Psychology                           13.8%
Law                                  10.7%
Business & Economics                 10.7%
Education                             5.7%
Language Arts & Disciplines           4.3%
Medical                               2.0%
History                               1.6%
Philosophy                            0.8%
NONE                                  0.7%
Family & Relationships                0.7%
Computers                             0.5%
Architecture                          0.4%
Science                               0.4%
Performing Arts                       0.4%
Music                                 0.3%
Sports & Recreation                   0.3%
Nature                                0.3%
Technology & Engineering              0.2%
Religion                              0.1%
Health & Fitness                      0.1%
Art                                   0.1%
Tra

>The sample size of 40 was chosen based on a power analysis using the Wilson method for estimating binomial proportion confidence intervals (CIs). Observing 40 consecutive acceptable ratings provides a 95% CI of 95.6% ± 4.4%, ensuring a lower bound of over 90% for the true proportion of acceptable cases.

Confidence intervals were calculated using [Proportion Confidence Interval Calculator](https://www.statskingdom.com/proportion-confidence-interval-calculator.html).

>Readability formulas such as FKGL and FRE assume continuous prose. When the selected text deviates significantly from typical prose, such as glossary entries, answer-key lists, or structured outlines, these formulas can yield extreme, uninformative values. For example, an extended run-on “sentence” created by a bulleted list of phrases lacking punctuation can artificially inflate FKGL or sharply decrease FRE scores, resulting in misleading values unrelated to the simplification tool’s actual performance. Manual inspection revealed that virtually all passages assessed in the top 1% in reading difficulty by either metric (FKGL > 44.0 or FRE < -60.9) represented these non-prose formats. These outliers (_n_ = 634, 1.2% of the dataset) were therefore excluded from analysis.

In [5]:
fkgl_high = events.rd_sel_fkgl.quantile( .99 )
fre_low = events.rd_sel_fre.quantile( .01 )
print( f'Outliers: FKGL > {fkgl_high:.1f} or FRE < {fre_low:.1f}' )

Outliers: FKGL > 44.0 or FRE < -60.9


In [6]:
remove = ( events.rd_sel_fkgl > fkgl_high ) | ( events.rd_sel_fre < fre_low )
print( f'{remove.sum()} outlier events ({remove.mean():.1%}) removed' )

634 outlier events (1.2%) removed


>Re-running the full dataset without trimming altered the mean FKGL improvement by ~0.5 grade levels and the mean FRE improvement by less than 2 points (but considerably reduced standard deviations), with no change to the overall statistical conclusions.

In [7]:
before = events.rd_sel_fkgl.describe()
after = events[ ~remove ].rd_sel_fkgl.describe()
stats = pd.concat( [
    before,
    after,
    after - before,
], axis=1 ).round( 2 ).set_axis( [ 'before', 'after', 'Δ' ], axis=1 )
stats.index.set_names( 'rd_sel_fkgl', inplace=True )
stats

,before,after,Δ
rd_sel_fkgl,,,
count,54371.00,53737.00,-634.00
mean,17.15,16.65,-0.50
std,7.12,4.86,-2.26
min,-0.53,-0.53,0.00
25%,13.56,13.53,-0.03
50%,16.03,15.97,-0.06
75%,18.99,18.86,-0.13
max,210.70,44.03,-166.66


In [8]:
before = events.rd_sel_fre.describe()
after = events[ ~remove ].rd_sel_fre.describe()
stats = pd.concat( [
    before,
    after,
    after - before,
], axis=1 ).round( 2 ).set_axis( [ 'before', 'after', 'Δ' ], axis=1 )
stats.index.set_names( 'rd_sel_fre', inplace=True )
stats

,before,after,Δ
rd_sel_fre,,,
count,54371.00,53737.00,-634.00
mean,23.41,25.00,1.59
std,25.11,18.97,-6.14
min,-649.83,-60.82,589.01
25%,13.65,14.20,0.55
50%,25.98,26.24,0.26
75%,37.36,37.52,0.16
max,102.73,102.73,0.00


In [9]:
events = events[ ~remove ]

## 3. Results and Discussion

>**Table 1**<br/>Descriptive statistics for readability, lexical, syntactic, and semantic fidelity metrics for simplification events.

In [10]:
cols = events.columns[ -8: ]
events[ cols ].describe().round( 2 ).T

,count,mean,std,min,25%,50%,75%,max
rd_fkgl_difference,53737.0,-7.37,4.61,-39.33,-9.20,-6.53,-4.51,7.72
rd_fre_difference,53737.0,31.34,16.95,-37.41,20.18,29.33,40.07,152.26
lex_content_log_p_difference,53737.0,1.02,0.52,-2.08,0.66,0.98,1.33,5.63
lex_chars_per_word_difference,53737.0,-0.41,0.37,-3.70,-0.62,-0.38,-0.17,1.66
syn_dep_depth_difference,53737.0,-0.98,0.81,-14.25,-1.33,-0.87,-0.48,2.59
syn_words_per_sentence_difference,53737.0,-14.62,13.28,-261.50,-18.65,-11.45,-6.62,104.00
sem_cosine_similarity,53737.0,0.85,0.08,-0.03,0.81,0.87,0.91,0.99
sem_compression_ratio,53737.0,0.80,0.22,0.02,0.66,0.79,0.92,4.20


### 3.1. Readability

>Prior to simplification, the mean FKGL of selected textbook passages was 16.65, indicating content typically written at a level substantially above typical undergraduate reading expectations. The average simplification lowered FKGL by 7.37 grade levels to 9.28, bringing the text into a more accessible range for college-level readers. The interquartile range (Q<sub>1</sub> = -9.20, Q<sub>3</sub> = -4.51) shows that simplifications consistently resulted in meaningful readability improvements, with even the least-improved examples achieving several grade levels of improvement. The mean FRE increase was approximately 31 points (Q<sub>1</sub> = 20.18, Q<sub>3</sub> = 40.07), reinforcing that texts were easier to read post-simplification.

In [11]:
cols = [ col for col in events.columns if col.startswith( 'rd_' ) ]
events[ cols ].describe().round( 2 ).T

,count,mean,std,min,25%,50%,75%,max
rd_sel_fkgl,53737.0,16.65,4.86,-0.53,13.53,15.97,18.86,44.03
rd_sim_fkgl,53737.0,9.28,2.05,0.89,7.91,9.24,10.59,30.85
rd_sel_fre,53737.0,25.00,18.97,-60.82,14.20,26.24,37.52,102.73
rd_sim_fre,53737.0,56.34,12.34,-9.47,48.38,56.65,64.71,110.06
rd_fkgl_difference,53737.0,-7.37,4.61,-39.33,-9.20,-6.53,-4.51,7.72
rd_fre_difference,53737.0,31.34,16.95,-37.41,20.18,29.33,40.07,152.26


### 3.2. Lexical Simplification

>Lexical simplification was assessed through changes in word familiarity and length. The mean increase of 1.02 in the Δ log _p_ metric corresponds roughly to a 2.8-fold increase in average content word frequency. This indicates words were replaced with more common synonyms, increasing lexical familiarity for readers. The interquartile range (0.66 to 1.33) indicates consistent lexical simplifications. Simplified texts also showed an average reduction of 0.41 characters per word, suggesting a preference for shorter, simpler words.

In [12]:
cols = [ col for col in events.columns if col.startswith( 'lex_' ) ]
events[ cols ].describe().round( 2 ).T

,count,mean,std,min,25%,50%,75%,max
lex_sel_content_log_p,53737.0,-10.74,0.58,-16.95,-11.09,-10.72,-10.37,-7.92
lex_sim_content_log_p,53737.0,-9.71,0.59,-13.31,-10.08,-9.69,-9.32,-7.42
lex_sel_chars_per_word,53737.0,5.22,0.44,2.38,4.93,5.20,5.49,8.34
lex_sim_chars_per_word,53737.0,4.81,0.38,3.17,4.56,4.80,5.05,6.79
lex_content_log_p_difference,53737.0,1.02,0.52,-2.08,0.66,0.98,1.33,5.63
lex_chars_per_word_difference,53737.0,-0.41,0.37,-3.70,-0.62,-0.38,-0.17,1.66


### 3.3. Syntactic Simplification

>Dependency depth, reflecting syntactic complexity, was reduced on average by 0.98 levels, signaling less complex sentence structures. With an interquartile range from -1.33 to -0.48, the results indicate readers encounter fewer deeply embedded modifiers, which lowers working memory load and improves clarity. The average simplification reduced sentence length by about 14.6 words.

In [13]:
cols = [ col for col in events.columns if col.startswith( 'syn_' ) ]
events[ cols ].describe().round( 2 ).T

,count,mean,std,min,25%,50%,75%,max
syn_sel_dep_depth,53737.0,3.82,0.85,1.07,3.28,3.69,4.19,17.12
syn_sim_dep_depth,53737.0,2.84,0.43,1.07,2.55,2.79,3.08,7.50
syn_sel_words_per_sentence,53737.0,30.53,13.59,2.15,22.17,27.33,35.00,272.00
syn_sim_words_per_sentence,53737.0,15.91,3.53,5.67,13.67,15.54,17.67,151.00
syn_dep_depth_difference,53737.0,-0.98,0.81,-14.25,-1.33,-0.87,-0.48,2.59
syn_words_per_sentence_difference,53737.0,-14.62,13.28,-261.50,-18.65,-11.45,-6.62,104.00


### 3.4. Semantic Fidelity

>It is important to confirm that semantic fidelity (preservation of original meaning) is maintained alongside reductions in complexity. Mean cosine similarity was .85, suggesting that while the simplified texts differed structurally and lexically from their originals, the core meanings remained well-preserved. The narrow interquartile range (.81 to .91) indicates stable semantic fidelity across the majority of simplifications. Simplified texts on average retained 80% of the original length. The interquartile range (66% to 92%) shows variability, but consistently high ratios are consistent with simplifications that reduce extraneous complexity without losing critical information.

In [14]:
events[ [ 'sem_cosine_similarity', 'sem_compression_ratio' ] ].describe().round( 2 ).T

,count,mean,std,min,25%,50%,75%,max
sem_cosine_similarity,53737.0,0.85,0.08,-0.03,0.81,0.87,0.91,0.99
sem_compression_ratio,53737.0,0.80,0.22,0.02,0.66,0.79,0.92,4.20


>To further assess semantic fidelity, an empirical threshold for acceptable cosine similarity was established at .7 using the procedure described in the Method section. Applying this threshold, 94.5% of original-simplified pairs demonstrated acceptable fidelity. The remaining 5.5% warrant further investigation, as potential loss of meaning is indicated.

In [15]:
low_cosine = events.sem_cosine_similarity < 0.7
print( f'Acceptable fidelity = {1 - low_cosine.mean():.1%}' )

Acceptable fidelity = 94.5%


>First, however, it is important to recognize that a cosine similarity below .7 does not automatically indicate an unacceptable simplification. Of the 2,980 pairs with low cosine similarity, the majority (76.8%) are at least .6, i.e., only slightly below the threshold. The low-similarity cases were therefore divided into moderately low (≥ .6, _n_ = 2,289, 4.3% of the dataset) and very low (< .6, _n_ = 691, 1.3% of the dataset) similarity groups for further investigation.

In [16]:
low_cosine_events = events[ low_cosine ]
moderately_low_cosine = low_cosine_events.sem_cosine_similarity >= .6
very_low_cosine = low_cosine_events.sem_cosine_similarity < .6

In [17]:
print( f'{low_cosine.sum()} low-cosine pairs' )
print( f'{moderately_low_cosine.mean():.1%} have cosine ≥ .6' )
print( f'{moderately_low_cosine.sum()} moderately low pairs' )
print( f'{very_low_cosine.sum()} very low pairs' )

2980 low-cosine pairs
76.8% have cosine ≥ .6
2289 moderately low pairs
691 very low pairs


>Applying the previously established sampling method, 95% confidence intervals were calculated for the proportion of acceptable simplifications in each group. In the moderately low similarity group, 37 of 40 pairs were rated acceptable, resulting in a 95% CI of 88.8% ± 8.6%. In the very low group, 31 of 40 pairs were rated acceptable, giving a 95% CI of 75.1% ± 12.6%.

Confidence intervals were calculated using [Proportion Confidence Interval Calculator](https://www.statskingdom.com/proportion-confidence-interval-calculator.html).

#### 3.4.1. Example 1

>The following pair from a social science textbook (cosine similarity .55) illustrates how simplifications with relatively low cosine similarity can remain semantically acceptable.
>
>>Insofar as corrections remains at the heart of our social policy—rather than as a supplemental or marginal support as it was throughout most of United States history—it is the Iron State stealing from the future of the Golden State.
>>
>>When we focus too much on prisons as a main part of our social policy, it takes away from other important areas. In the past, prisons were just a small part of our approach. Now, they take up a lot of attention and resources. This focus on prisons is like taking away from our future growth and success.
>
>The simplification demonstrates substantial readability improvement, reducing FKGL from 19.0 to 7.2.

In [18]:
idx = 43083
row = events.loc[ idx ]
print( row.selected )
print()
print( row.simplified )
row.to_frame()

Insofar as corrections remains at the heart of our social policy—rather than as a supplemental or marginal support as it was throughout most of United States history—it is the Iron State stealing from the future of the Golden State. 

When we focus too much on prisons as a main part of our social policy, it takes away from other important areas. In the past, prisons were just a small part of our approach. Now, they take up a lot of attention and resources. This focus on prisons is like taking away from our future growth and success.


,43083
timestamp,2025-03-19 04:05:01
student_id,5RSRYJWT42D7GWBM6HYJ
textbook_id,9781544308074
subject,Social Science
selected,Insofar as corrections remains at the heart of...
simplified,When we focus too much on prisons as a main pa...
rd_sel_fkgl,18.984103
rd_sim_fkgl,7.154655
rd_sel_fre,28.419231
rd_sim_fre,69.593362


#### 3.4.2. Example 2

>We consider cases where the simplified text contains fewer than half the original number of words (CR < 0.5). Among pairs with low cosine similarity, a strong negative correlation was observed between the length of original selections and their CR (_r_ = -.72, _p_ < .001). This highlights that longer original passages were substantially more likely to undergo extensive summarization in simplification.

In [19]:
r, p = scipy.stats.pearsonr( low_cosine_events.sem_sel_words, low_cosine_events.sem_compression_ratio )
print( f'r = {r:.2f}, p = {p}' )

r = -0.72, p = 0.0


>Detailed examination of these pairs revealed notable trends. The primary strategy identified was summarization and condensation, going further than lexico-syntactic simplification for readability improvement. This approach led to significant reductions in supporting details, examples, historical context, nuanced definitions, and sometimes important qualifications originally present. Despite these substantial reductions, simplified versions generally retained accurate representations of the original core ideas. Semantic drift, although possible, was typically minimal, and hallucinations were not observed in these cases.
>
>The following simplification from a Psychology textbook (CR = 0.36, cosine similarity = .54) illustrates this pattern.

In [20]:
idx = 8986
row = events.loc[ idx ]
print( row.selected )
print()
print( row.simplified )
row.to_frame()

Reaction chains are similar to FAPs, but with one major difference—each set of responses in a reaction chain requires an appropriate stimulus to set it off. Recall that once a fixed-action pattern (FAP) begins, the animal usually continues the sequence even when the stimuli that set off the behavior are removed. In the previous squirrel and nuts example, the animal continues to dig a hole and bury the non-existent nut, even if the nut is removed. In contrast, a reaction chain requires the presence of a specific stimulus to activate each link in the sequence of behavior. An organism’s performance produces stimuli that set off the next series of responses in the chain; these behaviors in turn produce the stimuli followed by another set of responses. Presenting a stimulus that ordinarily occurs in the middle part of the sequence activates the chain at that point rather than at the beginning. Also, unlike FAPs, if the stimuli that activate behavior are removed, the sequence is disrupted.Th

,8986
timestamp,2024-11-27 02:54:28
student_id,JXVMZ366CHNWNCEBNFU2
textbook_id,9781000817225
subject,Psychology
selected,"Reaction chains are similar to FAPs, but with ..."
simplified,Reaction chains and fixed-action patterns (FAP...
rd_sel_fkgl,14.122994
rd_sim_fkgl,7.046447
rd_sel_fre,36.8284
rd_sim_fre,69.923882


#### 3.4.3. Example 3

>Very high compression ratios, where simplified texts contain over 50% more words than the original (CR > 1.5), were also analyzed. These cases, while less common than extensive summarization, have a main theme of elaboration and explanation: taking dense, complex, or technical information and significantly expanding it into simpler language. This process involves clarifying terms, providing context, adding examples, and explicitly stating ideas that were originally implicit or condensed. Instances of semantic drift are rare and generally mild when they occur. Genuine hallucination is essentially absent. Instead, the primary reason for the high compression ratio is typically due to the simplification process itself, which trades concise complexity for clear verbosity to enhance reader comprehension.
>
>An example simplification from a psychology textbook (CR = 2.03, cosine = .64) illustrates these concepts clearly:
>
>>Publication bias is a fundamental problem with our method of creating the official record of scientific inquiry such that studies that report statistically significant findings are more likely to be published than studies that report no statistically significant findings.
>>
>>When scientists do research, they often look for results that show a clear effect or change. These are called statistically significant findings. Studies with these kinds of results are more likely to get published in scientific journals. On the other hand, studies that don't find a clear effect, known as "negative studies," are less likely to be published. This creates a problem because it means that the official record of scientific research might not include all the important information.

In [21]:
idx = 13714
row = events.loc[ idx ]
print( row.selected )
print()
print( row.simplified )
row.to_frame()

Publication bias is a fundamental problem with our method of creating the official record of scientific inquiry such that studies that report statistically significant findings are more likely to be published than studies that report no statistically significant findings.

When scientists do research, they often look for results that show a clear effect or change. These are called statistically significant findings. Studies with these kinds of results are more likely to get published in scientific journals. On the other hand, studies that don't find a clear effect, known as "negative studies," are less likely to be published. This creates a problem because it means that the official record of scientific research might not include all the important information.


,13714
timestamp,2024-12-17 23:30:30
student_id,5P5CPVQJHXJTPKCHXQUC
textbook_id,9781000528824
subject,Psychology
selected,Publication bias is a fundamental problem with...
simplified,"When scientists do research, they often look f..."
rd_sel_fkgl,24.127692
rd_sim_fkgl,9.392253
rd_sel_fre,-8.457692
rd_sim_fre,55.866354
